# ZuCo paper analysis — compact resumable GPU stage
Run after **CPU PREPARATION COMPLETE**, on **T4 GPU**, using the same Drive account. The completed CPU preparation is reused; no source EEG download, cleaning or trial extraction is repeated.

**Drive storage:** this analysis has a **200 MB incremental allowance** for final results, compact permutation checkpoints and immutable model checkpoints. It also enforces the original saved workflow cap. The large random permutation maps stay in runtime RAM; only the exact exceedance counts and cluster maxima required by the tests are saved. Individual EEG trials/features remain unchanged.

**Resume:** completed batches persist on Drive and are skipped after a disconnect. Only an unfinished batch (up to 25 permutations) needs repeating. Model fitting also resumes from saved small checkpoints. Loading saved trial archives into a new runtime can take time; this is not re-extraction. Do not run two analyses into the same result folder.

The CPU notebook remains pinned to its existing scientific implementation. This newer analysis launcher verifies that implementation and the saved reader checksums before proceeding. If an old full-map GPU workspace is detected, it stops for review rather than deleting anything.


## Connect Drive and load the pinned code
Choose the same account for Drive mount and quota authentication. No GitHub token is required.

In [ ]:
from google.colab import drive, auth
from pathlib import Path
import os, sys, subprocess, json
from IPython.display import display, Markdown, Image
import pandas as pd

drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
OUT=THESIS/'Results/word_sentiment_eeg/zuco_paper_figures_recovered'
REPO=Path('/content/word-sentiment-eeg-paper-recovered-analysis')
CODE_REVISION='310d220befca0333ab8e65ce7b6585866854cb52'
def git(*args):
    result=subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],
                          text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                          env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
    if result.returncode:
        raise RuntimeError('Git setup failed:\n'+result.stdout)
    if result.stdout:print(result.stdout,end='',flush=True)
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary checkout contains edits. Preserve them before updating.')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    git('-C',REPO,'fetch','--depth','1','origin',CODE_REVISION)
    fetched=subprocess.check_output(['git','-C',str(REPO),'rev-parse','FETCH_HEAD^{commit}'],text=True).strip()
    if fetched!=CODE_REVISION:raise RuntimeError('Fetched code differs from the pinned commit.')
    git('-C',REPO,'checkout','--detach','FETCH_HEAD')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    raise RuntimeError('Code checkout verification failed.')
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-paper.txt')],check=True)
print('Code revision:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

# A mounted Drive filesystem disk_usage value can reflect the runtime volume;
# use the authenticated Drive quota instead.
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials,_=google.auth.default()
response=AuthorizedSession(credentials).get('https://www.googleapis.com/drive/v3/about',
    params={'fields':'storageQuota,user(emailAddress)'},timeout=30)
response.raise_for_status();quota=response.json()['storageQuota']
print('Quota account:',response.json().get('user',{}).get('emailAddress','unavailable'))
if 'limit' not in quota:
    raise RuntimeError('Drive did not expose a finite storage quota; verify free space before starting.')
FREE_DRIVE_BYTES=max(0,int(quota['limit'])-int(quota.get('usage',0)))
print(f'Account free space: {FREE_DRIVE_BYTES/1e9:.2f} GB. Workflow checks the additional space needed before source reads.')
print('Destination:',OUT)


## Load prepared trials and run analysis

In [ ]:
from datetime import datetime, timezone
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[sys.executable,'-u',str(REPO/'scripts/analyze_paper_small_storage.py'),
         '--thesis-root',str(THESIS),'--dataset','zuco','--free-drive-bytes',str(FREE_DRIVE_BYTES)]
attempt=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
run_with_progress(command,OUT/'status'/f'compact_analyze_{attempt}.json',activity='ZuCo compact GPU analysis using saved trials')


## Figure 1 — Selection and cleaning

In [ ]:
display(Image(filename=str(OUT/'figures/figure_1_selection_cleaning.png')))


## Figure 2 — Raw FRPs

In [ ]:
display(Image(filename=str(OUT/'figures/figure_2_raw_frp.png')))


## Figure 3 — Cleaned FRPs

In [ ]:
display(Image(filename=str(OUT/'figures/figure_3_cleaned_frp.png')))


## Figure 4 — Controls

In [ ]:
display(Image(filename=str(OUT/'figures/figure_4_controls.png')))


## Coverage, exclusions and methods

In [ ]:
display(pd.read_csv(OUT/'analysis/source_coverage.csv'))
display(pd.read_csv(OUT/'analysis/exclusion_counts.csv'))
print((OUT/'analysis/analysis_manifest.json').read_text())
display(Markdown((OUT/'METHOD_DIFFERENCES.md').read_text()))
